# Running Existing MCP Servers

The **Model Context Protocol** lets an agent use tools that live in a separate process — a server you didn't write and don't have to import. This works through the transports MCP defines, then connects to a real server and calls its tools.

Covers the stdio and HTTP transports, the FastMCP client, listing a server's tools and inspecting their schemas, then calling them. `multi_server_agent.py` puts it together: a LangGraph ReAct agent wired to two MCP servers at once.

## What this covers

- How stdio, stdout and stderr carry an MCP conversation
- Connecting to an MCP server over stdio and over HTTP
- Listing a server's tools and reading their input schemas
- Calling a tool and handling the result
- Wiring multiple MCP servers into one agent

Only one library is needed for the notebook itself.

In [ ]:
%%capture
%pip install fastmcp

## How MCP talks: standard streams

The stdio transport runs the server as a subprocess and exchanges JSON over its standard streams. A quick refresher on those first.

In [ ]:
my_code="hello"
print(my_code)

`sys.stdout.write()` is the primitive `print` builds on; it returns the character count.

In [ ]:
import sys
sys.stdout.write("Hello")

### stdin

The input side of the same pair.

In [ ]:
# In a real MCP exchange stdin carries JSON from the client. Here we just
# assign a value directly so the notebook runs without waiting for input.
name = "Ada"
name

In [ ]:
print(name)

### stderr

A separate stream, so diagnostics never corrupt the protocol traffic on stdout.

In [ ]:
# Normal output goes to stdout
print("This is standard output (stdout)")

# Error messages can be written to stderr
print("This is an error message (stderr)", file=sys.stderr)

# Example: catching a real error
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Error occurred: {e}", file=sys.stderr)

## The client

`fastmcp` provides the client and the transports.

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport, StdioTransport

## Connecting to a server

[Context7](https://context7.com) serves up-to-date library documentation over MCP — a good example of a server you consume rather than build.

### stdio transport

Runs the server as a subprocess (needs Node and `npx` on PATH).

In [ ]:
stdio_transport = StdioTransport(
    command = "npx",
    args=["-y", "@upstash/context7-mcp"]
)
print(stdio_transport )

The transport takes the command, its arguments, and the environment to run it in.

### Client

Wraps a transport and manages the session.

In [ ]:
stdio_client = Client(stdio_transport)

Opening the client as a context manager starts the server and closes it cleanly afterwards.

In [ ]:
async with stdio_client as client:
    # List of tools the server provides
    tools = await client.list_tools()

print("Done")

In [ ]:
len(tools)

Each tool object carries a name, a description and an input schema.

In [ ]:
print(tools[0].name)

`resolve-library-id` maps a library name to the identifier the docs tool expects.

In [ ]:
print(tools[0].description)

`inputSchema` declares the arguments — this is what the model reads to call the tool correctly.

In [ ]:
tools[0].inputSchema

The second tool fetches the documentation itself.

In [ ]:
print(
f""" name: {tools[1].name}: \n
description: {tools[1].description} \n
inputSchema: {tools[1].inputSchema}""")

### Calling a tool

`call_tool` runs it and returns the result.

Let's use the context manager again to call the `resolve-library-id` tool and search for `fastmcp` documentation.


## call_tool

In [ ]:
async with stdio_client as client:
    # Find a library ID via a search query
    response = await client.call_tool("resolve-library-id", {
        "libraryName": "fastmcp",
        "query": "I want to create a new MCP server using the fastmcp Python framework"
    })

print(response.content[0].text)

When you call the `resolve-library-id` tool, it returns a list of matching libraries for your search term. The output shows multiple options with details to help you choose the best one. Each result includes a Library ID (the unique identifier you'll need for the next step), a description of what the library does, the number of code snippets available, and a benchmark score indicating reliability.

For example, we will select the following result:

```
- Title: FastMCP
- Context7-compatible library ID: /llmstxt/gofastmcp_llms-full_txt
- Description: FastMCP is a Python framework for building MCP servers, featuring an experimental OpenAPI parser, JWT claims, and optimized payload handling.
- Code Snippets: 12289
- Source Reputation: High
- Benchmark Score: 79
```

Taking a look at the list, we'll choose the library with the highest **Benchmark Score**. The **Context7-compatible library ID** `/llmstxt/gofastmcp_llms-full_txt` has a Trust Score of 9.6 so we'll go with that one.



### query-docs

Now we'll use the `query-docs` tool to retrieve the actual documentation for that specific library.




We'll use the context manager once again to fetch the code snippets and documentation of `/llmstxt/gofastmcp_llms-full_txt` using the `query-docs` tool.


In [ ]:
async with stdio_client as client:
    # Use resolved ID to fetch documentation
    docs = await client.call_tool("query-docs", {
        "libraryId": "/llmstxt/gofastmcp_llms-full_txt",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
    })

    print(docs.content[0].text[:1000])

As you can see, we have printed out the first 1000 characters of the library documentation of `/punkpeye/fastmcp`. This documentation is what LLMs and AI code agents use to keep their knowledge on common libraries and frameworks up-to-date.


###  Question 1.  How do you use the resolve-library-id tool to find the library ID for scikit-learn



### Question 2. How do you get the actual documentation once you have the library ID?


### HTTP Preface

The HTTP protocol allows data to be transferred over the internet. Since the protocol can be somewhat complex, we will focus on the GET method, one of the most common HTTP methods.

The figure below illustrates a typical response. The response start line contains:

- Version number (e.g., HTTP/1.0)

- Status code (e.g., 200 for success)

- Descriptive phrase (e.g., OK)

The response header follows, providing useful metadata. Finally, the response body includes the requested resource, such as an HTML document. It should also be noted that some requests include headers, which provide additional details or parameters for the transaction.

Requests is a Python Library that allows you to send `HTTP/1.1` requests easily. We can import the library as follows:


In [ ]:
import requests

You can make a `GET` request via the method `get` to [example.com](https://example.com):


In [ ]:
url='https://www.ibm.com/'
r=requests.get(url)

We have the response object `r`, this has information about the request, such as the status of the request. We can view the status code using the attribute `status_code`.


In [ ]:
r.status_code

You can view the request headers and the body (there is no body for a get request so we get 'None'):


In [ ]:
print(r.request.headers)
print("request body:", r.request.body)

### HTTP Transport

Now let's talk to the Context7 MCP server over HTTPS—no local process needed. The HTTP transport also acts like a bridge, but instead of connecting from your notebook/Python file to your local computer, it connects to a remote server. This is ideal when the server is hosted remotely or you don't want to manage a subprocess. We'll use the same Client API with a different "wire" (transport). Once you create the transport, the process is pretty much identical.

<pre style="text-align: center;">
Your Python Code  ←→  [http_transport]  ←→  Context7 Server (remote)
   (Client)              (Bridge)              (Context7's servers)
</pre>

First, we create the `StreamableHttpTransport` object. 


In [ ]:
http_transport = StreamableHttpTransport(
    url="https://mcp.context7.com/mcp"
)

- `url`: The server's MCP endpoint that exposes all the tools, prompts, and resources
- “Streamable” means responses can arrive incrementally; Client handles this for you.

Let's wrap the transport in a Client.


In [ ]:
http_client = Client(http_transport)

Finally, like before, let's list and call the tools with the same configuration as above.


In [ ]:
async with http_client as client:
    tools = await client.list_tools()

    response = await client.call_tool("resolve-library-id", {
        "libraryName": "fastmcp",
        "query": "I want to create a new MCP server using the fastmcp Python framework"
    })

    docs = await client.call_tool("query-docs", {
        "libraryId": "/llmstxt/gofastmcp_llms-full_txt",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
    })

Let's check by printing out the outputs, they should match to the outputs above.


In [ ]:
for tool in tools:
        print(
f"""{tool.name}: \n
{tool.description} \n
{tool.inputSchema}""")
print(response.content[0].text[:1000])
print(docs.content[0].text[:500]) 

## Other Transports

There are other transports but we won't demo them here as they are not as widely used as STDIO or HTTP. These two are enough for most use cases of MCP servers.

### SSE Transport

The Server-Sent Events transport enables HTTP communication between MCP servers and clients using an asymmetric channel. It uses SSE for server-to-client streaming and HTTP POST for client-to-server messages. It was replaced by the Streamable HTTP transport to provide bidirectional streaming capabilities and improved real-time communication efficiency.

Read more [here](https://gofastmcp.com/clients/transports#sse-transport-legacy).

### In-Memory Transport

In-memory transport connects a client directly to a FastMCP server instance within the same Python process. This eliminates network overhead by enabling direct function calls between client and server components. 

Read more [here](https://gofastmcp.com/clients/transports#in-memory-transport).


## Referenced Sources

[FastMCP](https://gofastmcp.com/getting-started/welcome) <br>
[Context7](https://context7.com/)


## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)